# Evaluation Metrics & Cross-Validation

Practice activity from Microsoft *Foundations of AI and Machine Learning* — Module: AI/ML Algorithms and Techniques.

Goal: compare a single train-test split against k-fold cross-validation, on both a classification task and a regression task.

## 2. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    make_scorer,
)

## 3. Load and prepare the data

In [2]:
data = {
    'StudyHours':    [1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    'PrevExamScore': [30, 40, 45, 50, 60, 65, 70, 75, 80, 85],
    'Pass':          [0,  0,  0,  0,  0,  1,  1,  1,  1,  1],
}
df = pd.DataFrame(data)

X = df[['StudyHours', 'PrevExamScore']]
y = df['Pass']
df

,StudyHours,PrevExamScore,Pass
0,1,30,0
1,2,40,0
2,3,45,0
3,4,50,0
4,5,60,0
5,6,65,1
6,7,70,1
7,8,75,1
8,9,80,1
9,10,85,1


## 4. Evaluation metrics on a single train-test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print(f'Accuracy:  {accuracy}')
print(f'Precision: {precision}')
print(f'Recall:    {recall}')
print(f'F1-Score:  {f1}')

Accuracy:  1.0
Precision: 1.0
Recall:    1.0
F1-Score:  1.0


A single split gives one number per metric — fine as a sanity check, but the score depends entirely on which 2 rows landed in the test set.

## 5. Why cross-validation?

Cross-validation rotates which subset of the data is held out, so the reported score reflects the model rather than a lucky split. We use **k-fold**: the data is partitioned into *k* equal folds; each fold takes a turn as the test set while the rest train the model.

## 6. K-fold cross-validation (accuracy)

In [4]:
model = LogisticRegression()
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f'Per-fold accuracies: {cv_scores}')
print(f'Mean accuracy:       {cv_scores.mean():.4f}')

Per-fold accuracies: [1.  1.  1.  1.  0.5]
Mean accuracy:       0.9000


## 7. Cross-validation with multiple metrics

`cross_validate` accepts a dict of scorers. We wrap precision/recall/F1 with `make_scorer(..., zero_division=0)` so they stay defined when a fold has only one class.

In [5]:
scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall':    make_scorer(recall_score,    zero_division=0),
    'f1':        make_scorer(f1_score,        zero_division=0),
}
cv_results = cross_validate(model, X, y, cv=5, scoring=scoring)

for metric in scoring:
    key = f'test_{metric}'
    print(f'{metric.capitalize():10s} per-fold: {cv_results[key]}  mean: {cv_results[key].mean():.4f}')

Accuracy   per-fold: [1.  1.  1.  1.  0.5]  mean: 0.9000
Precision  per-fold: [1.  1.  1.  1.  0.5]  mean: 0.9000
Recall     per-fold: [1. 1. 1. 1. 1.]  mean: 1.0000
F1         per-fold: [1.         1.         1.         1.         0.66666667]  mean: 0.9333


## 8. Cross-validation with a regression model

Now predict `PrevExamScore` from `StudyHours` with linear regression. Scikit-learn returns the **negative** of MSE/MAE so that across all scorers "higher is better," which is why we flip the sign when printing.

In [6]:
X_reg = df[['StudyHours']]
y_reg = df['PrevExamScore']
reg_model = LinearRegression()

reg_scoring = {
    'r2':      'r2',
    'neg_mse': 'neg_mean_squared_error',
    'neg_mae': 'neg_mean_absolute_error',
}
reg_results = cross_validate(reg_model, X_reg, y_reg, cv=5, scoring=reg_scoring)

print(f"R-squared per-fold: {reg_results['test_r2']}  mean: {reg_results['test_r2'].mean():.4f}")
print(f"MSE per-fold:       {-reg_results['test_neg_mse']}  mean: {-reg_results['test_neg_mse'].mean():.4f}")
print(f"MAE per-fold:       {-reg_results['test_neg_mae']}  mean: {-reg_results['test_neg_mae'].mean():.4f}")

R-squared per-fold: [ 0.52933673  0.88503086 -0.60298929  0.88503086 -1.28939909]  mean: 0.0814
MSE per-fold:       [11.76658163  0.7185571  10.01868308  0.7185571  14.30874433]  mean: 7.5062
MAE per-fold:       [2.67857143 0.69444444 3.125      0.69444444 3.7202381 ]  mean: 2.1825


## Key takeaways

- A single 80/20 split gives one accuracy number that depends entirely on which 2 rows land in the test set. With 10 rows, that's high variance.
- K-fold rotates the test fold *k* times and averages, so the score reflects the model rather than a lucky split.
- For **classification**, average accuracy/precision/recall/F1 across folds. Watch for ill-defined metrics on folds with one class — use `zero_division=0` to keep them numeric.
- For **regression**, use R-squared plus MAE/MSE. Remember sklearn returns the *negative* MSE/MAE so that "higher is better" is consistent across scorers.
- R-squared can come back **negative** on tiny per-fold test sets where the variance is small — that's expected on a 10-row dataset, not a bug. It means the model did worse than predicting the mean on that fold.